In [1]:
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
import matplotlib.pyplot as plt

from transformers import pipeline
from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

2024-10-21 00:33:12.136080: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
start_date = '20241010'
end_date = '20241010'

In [4]:
click_customer = f'''
    WITH response AS (
    
    SELECT
        yyyymmdd,
        userid,
        -- appversion,
        -- responsestatus,
        -- adid,
        aduuid,
        responsestatus,
        -- coalesce(aduuid, adid) unique_id,
        REPLACE(REGEXP_EXTRACT(metadata1, 'title=([^,]*)'), 'title=', '') AS title,
        REPLACE(REGEXP_EXTRACT(metadata1, 'advertiserName=([^,]*)'), 'advertiserName=', '') AS advertiserName
    
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND responseType = 'GAMBanner'
        AND eventName = 'Ad_Response'
        AND responsestatus != 'false'
        AND appversion >= '8.30.0'
    
    ),
    
    click AS (
    
    SELECT
        yyyymmdd,
        userid,
        aduuid
    
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND responseType = 'GAMBanner'
        AND eventName = 'Ad_Click'
        AND appversion >= '8.30.0'
        AND aduuid IS NOT NULL
    )
    
    SELECT
        response.userid,
        response.title,
        response.advertiserName
    FROM 
        response
    JOIN 
        click
        on response.yyyymmdd = click.yyyymmdd
        and response.userid = click.userid
        and response.aduuid = click.aduuid
    WHERE 
        response.advertiserName IS NOT NULL OR response.title IS NOT NULL
    GROUP BY 1,2,3
    
'''

df = pd.read_sql(click_customer, connection)

In [5]:
df.head(5)

,userid,title,advertiserName
0,661c291af86042ebf28fdd8c,Naye Bharat Ka Naya Index,Axis Mutual Fund
1,6246a84088d8cd693f572fab,Samsung Galaxy M55s 5G,Samsung
2,64eebb16710fe8060b55e688,Bigg Boss Season 8,Disney + Hotstar
3,65723e9428808d64f839cc89,Samsung Galaxy M55s 5G,Samsung
4,6416feae20601dd0d331509a,Bigg Boss Season 8,Disney + Hotstar


In [57]:
# Categories for classification
CATEGORIES = [
    'Finance','Technology','Entertainment','Automotive','Healthcare','Retail/E-commerce','Travel & Tourism',
    'Food & Beverage', 'Real Estate','Education','Beauty & Personal Care','Consumer Goods','Telecommunications',
    'Sports & Recreation', 'Social Network & Media','Fashion & Apparel','Home & Garden','Legal Services',
    'Logistics & Delivery Services', 'Gaming & Esports','Media & Publishing','Business Services','Energy & Utilities',
    'Agriculture','Luxury Goods & Services', 'Events & Conferences','Parenting & Childcare','Pets & Animals',
    'DIY & Crafts','Spirituality & Religion', 'Automobile', 'Unknown'
]

# Updated list of models to compare
MODELS = [
    "valhalla/distilbart-mnli-12-1", 
    # "facebook/roberta-large-mnli",
    # "facebook/roberta-base-mnli",
    # "typeform/bert-base-uncased-mnli",
    "cross-encoder/nli-distilroberta-base"
    # "joeddav/xlm-roberta-large-xnli"       
]

def clean_columns(df):
    df['title'] = df['title'].replace('', None)
    df['advertiserName'] = df['advertiserName'].replace('', None)
    df['key_word_title'] = df['title'].combine_first(df['advertiserName'])  # Fill NaNs
    df['key_word_advertiser_name'] = df['advertiserName'].combine_first(df['title'])  # Fill NaNs
    return df

def initialize_classifier(model_name):
    device = 0 if torch.cuda.is_available() else -1  # Check if GPU is available
    classifier = pipeline("zero-shot-classification", model=model_name, device=device)
    return classifier

def classify_advertiser(key_word, classifier):
    try:
        result = classifier(key_word, CATEGORIES)
        return result['labels'][0]  # Return the category with the highest score
    except Exception as e:
        print(f"Error classifying '{key_word}': {e}")
        return 'Unknown'

def classify_and_merge(df, classifier):
    df_unique = df[['title', 'advertiserName', 'key_word_title', 'key_word_advertiser_name']].drop_duplicates().reset_index(drop=True)
    df_unique['category_title'] = df_unique['key_word_title'].apply(lambda x: classify_advertiser(x, classifier))
    
    unknown_mask = df_unique['category_title'] == 'Unknown'
    df_unique['category_advertiser_name'] = np.where(
        unknown_mask,
        df_unique['key_word_advertiser_name'].apply(lambda x: classify_advertiser(x, classifier)),
        df_unique['category_title']
    )
    
    df_unique['category'] = np.where(
        df_unique['category_title'] == 'Unknown', 
        df_unique['category_advertiser_name'], 
        df_unique['category_title']
    )
    
    df_merged = pd.merge(df, df_unique, how='left', on=['title', 'advertiserName'])
    return df_merged

def evaluate_models(df):
    results = {}
    for model_name in MODELS:
        print(f"Evaluating model: {model_name}")
        classifier = initialize_classifier(model_name)
        
        df_final = classify_and_merge(df, classifier)
        
        # Store the results in a dictionary with the model name as the key
        results[model_name] = df_final[['title', 'advertiserName', 'category']].copy()
        
        # Optionally, display the first few results
        print(f"Results from {model_name}:")
        print(results[model_name].head())
        
    return results

def main(df):
    df_cleaned = clean_columns(df)
    results = evaluate_models(df_cleaned)
    return results

In [58]:
df_final = main(df)  

Evaluating model: valhalla/distilbart-mnli-12-1
Results from valhalla/distilbart-mnli-12-1:
                       title    advertiserName            category
0  Naye Bharat Ka Naya Index  Axis Mutual Fund             Finance
1     Samsung Galaxy M55s 5G           Samsung  Telecommunications
2         Bigg Boss Season 8  Disney + Hotstar       Entertainment
3     Samsung Galaxy M55s 5G           Samsung  Telecommunications
4         Bigg Boss Season 8  Disney + Hotstar       Entertainment
Evaluating model: cross-encoder/nli-distilroberta-base
Results from cross-encoder/nli-distilroberta-base:
                       title    advertiserName       category
0  Naye Bharat Ka Naya Index  Axis Mutual Fund        Unknown
1     Samsung Galaxy M55s 5G           Samsung     Technology
2         Bigg Boss Season 8  Disney + Hotstar  Entertainment
3     Samsung Galaxy M55s 5G           Samsung     Technology
4         Bigg Boss Season 8  Disney + Hotstar  Entertainment


In [59]:
df_final

{'valhalla/distilbart-mnli-12-1':                           title    advertiserName            category
 0     Naye Bharat Ka Naya Index  Axis Mutual Fund             Finance
 1        Samsung Galaxy M55s 5G           Samsung  Telecommunications
 2            Bigg Boss Season 8  Disney + Hotstar       Entertainment
 3        Samsung Galaxy M55s 5G           Samsung  Telecommunications
 4            Bigg Boss Season 8  Disney + Hotstar       Entertainment
 ...                         ...               ...                 ...
 2446     Samsung Galaxy M55s 5G           Samsung  Telecommunications
 2447     Samsung Galaxy M55s 5G           Samsung  Telecommunications
 2448         Bigg Boss Season 8  Disney + Hotstar       Entertainment
 2449               Suzuki Bikes      Suzuki Bikes          Automobile
 2450     Samsung Galaxy M55s 5G           Samsung  Telecommunications
 
 [2451 rows x 3 columns],
 'cross-encoder/nli-distilroberta-base':                           title    advertiserN

In [60]:
def initialize_classifier_final():
    device = 0 if torch.cuda.is_available() else -1  # Check if GPU is available
    classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-1", device=device)
    return classifier

def classify_advertiser_final(key_word, classifier):
    try:
        result = classifier(key_word, CATEGORIES)
        return result['labels'][0]  # Return the category with the highest score
    except Exception as e:
        print(f"Error classifying '{key_word}': {e}")
        return 'Unknown'

def classify_and_merge_final(df, classifier):
    df_unique = df[['title', 'advertiserName', 'key_word_title', 'key_word_advertiser_name']].drop_duplicates().reset_index(drop=True)
    df_unique['category_title'] = df_unique['key_word_title'].apply(lambda x: classify_advertiser_final(x, classifier))
    
    unknown_mask = df_unique['category_title'] == 'Unknown'
    df_unique['category_advertiser_name'] = np.where(unknown_mask,
                                                      df_unique['key_word_advertiser_name'].apply(lambda x: classify_advertiser_final(x, classifier)),
                                                      df_unique['category_title'])
    
    df_unique['category'] = np.where(df_unique['category_title'] == 'Unknown', 
                                      df_unique['category_advertiser_name'], 
                                      df_unique['category_title'])
    
    df_merged = pd.merge(df, df_unique, how='left', on=['title', 'advertiserName'])
    return df_merged

def main_final(df):
    df_cleaned = clean_columns(df)
    classifier = initialize_classifier_final()
    df_final = classify_and_merge_final(df_cleaned, classifier)
    return df_final

In [61]:
df_category_tag = main_final(df)  

In [62]:
df_category_tag

,userid,title,advertiserName,key_word,key_word_title_x,key_word_advertiser_name_x,key_word_title_y,key_word_advertiser_name_y,category_title,category_advertiser_name,category
0,661c291af86042ebf28fdd8c,Naye Bharat Ka Naya Index,Axis Mutual Fund,Naye Bharat Ka Naya Index,Naye Bharat Ka Naya Index,Axis Mutual Fund,Naye Bharat Ka Naya Index,Axis Mutual Fund,Unknown,Finance,Finance
1,6246a84088d8cd693f572fab,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
2,64eebb16710fe8060b55e688,Bigg Boss Season 8,Disney + Hotstar,Bigg Boss Season 8,Bigg Boss Season 8,Disney + Hotstar,Bigg Boss Season 8,Disney + Hotstar,Unknown,Entertainment,Entertainment
3,65723e9428808d64f839cc89,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
4,6416feae20601dd0d331509a,Bigg Boss Season 8,Disney + Hotstar,Bigg Boss Season 8,Bigg Boss Season 8,Disney + Hotstar,Bigg Boss Season 8,Disney + Hotstar,Unknown,Entertainment,Entertainment
...,...,...,...,...,...,...,...,...,...,...,...
2446,62d6a7a5fe03f9f475c5e442,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
2447,6706bfd06d57faf726fc74b8,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung Galaxy M55s 5G,Samsung,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
2448,5fc52a75808c7a1abf171f12,Bigg Boss Season 8,Disney + Hotstar,Bigg Boss Season 8,Bigg Boss Season 8,Disney + Hotstar,Bigg Boss Season 8,Disney + Hotstar,Unknown,Entertainment,Entertainment
2449,63831f5a39691bc47e5b2a8a,Suzuki Bikes,Suzuki Bikes,Suzuki Bikes,Suzuki Bikes,Suzuki Bikes,Suzuki Bikes,Suzuki Bikes,Automobile,Automobile,Automobile


In [63]:
df_category_tag[['userid', 'title', 'advertiserName', 'category_title', 'category_advertiser_name', 'category']]

,userid,title,advertiserName,category_title,category_advertiser_name,category
0,661c291af86042ebf28fdd8c,Naye Bharat Ka Naya Index,Axis Mutual Fund,Unknown,Finance,Finance
1,6246a84088d8cd693f572fab,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
2,64eebb16710fe8060b55e688,Bigg Boss Season 8,Disney + Hotstar,Unknown,Entertainment,Entertainment
3,65723e9428808d64f839cc89,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
4,6416feae20601dd0d331509a,Bigg Boss Season 8,Disney + Hotstar,Unknown,Entertainment,Entertainment
...,...,...,...,...,...,...
2446,62d6a7a5fe03f9f475c5e442,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
2447,6706bfd06d57faf726fc74b8,Samsung Galaxy M55s 5G,Samsung,Telecommunications,Telecommunications,Telecommunications
2448,5fc52a75808c7a1abf171f12,Bigg Boss Season 8,Disney + Hotstar,Unknown,Entertainment,Entertainment
2449,63831f5a39691bc47e5b2a8a,Suzuki Bikes,Suzuki Bikes,Automobile,Automobile,Automobile
